        # 🐍 L01　Python 初體驗
        **Python 冒險之旅 2026**　｜　Day 1（08/29 六）🏝️ 起始之島　｜　關卡　｜　🏅 100 XP

        📖 對應教科書：第 1 章 1.3、1.7


        ### 🎯 這一關你會學到
        - 用 print() 顯示文字與數字
- 看懂並修正常見錯誤訊息
- 使用註解說明程式

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/python-quest-2026/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  Python 冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins

_LEVEL = "L01"
_SALT = "python-quest-2026-datama"
_TASKS = ["1-1", "1-2", "1-3", "1-4"]
_XP_EACH = 25
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_pyquest_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

def 行列表(out):
    return [ln.rstrip() for ln in str(out).splitlines() if ln.strip()]

class _NeedMoreInput(Exception):
    pass

_HIST = builtins.__dict__.setdefault("_pyquest_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_pyquest_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_pyquest_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

def _find_cell(tid):
    marker = "# 🎯 任務 " + tid
    for cell in reversed(_history()):
        if marker in cell:
            lines = [ln for ln in cell.splitlines()
                     if not re.match(r"\s*(檢查|通關密語)\s*\(", ln)]
            return "\n".join(lines)
    return None

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                _plt.show = _orig_show
        return buf.getvalue(), ns
    run.src = src
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _progress():
    done = sum(1 for t in _TASKS if _PASSED.get(t))
    bar = "■" * done + "□" * (len(_TASKS) - done)
    return f"[{bar}] {done}/{len(_TASKS)}"

def 檢查(tid):
    tid = str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    src = _find_cell(tid)
    if src is None:
        print(f"❌ 找不到「# 🎯 任務 {tid}」的程式格。請先執行那一格（並保留第一行的標記），再執行這裡。")
        return
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        result = (False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。")
    except Exception as e:
        tb = traceback.format_exc().strip().splitlines()[-1]
        result = (False, f"程式執行時發生錯誤 → {tb}")
    ok, extra = (result, "") if isinstance(result, bool) else result
    if ok:
        first = not _PASSED.get(tid)
        _PASSED[tid] = True
        print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")
        if all(_PASSED.get(t) for t in _TASKS):
            print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
    else:
        print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
        if extra: print("   💬 " + str(extra))
        if _HINTS.get(tid): print("   💡 提示：" + _HINTS[tid])
        print("   👉 修改程式後，先重新執行任務那一格，再執行這一格。")

def 通關密語():
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_SALT}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：PYQ-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_1_1(run):
    out, ns = run()
    need = ["床前明月光", "疑是地上霜", "舉頭望明月", "低頭思故鄉"]
    lines = 行列表(out)
    if not 出現(out, *need):
        return (False, "四句都要印出來，檢查有沒有錯字。")
    return (len(lines) >= 4, "要用 4 個 print()，每句各印一行。")
任務定義("1-1", _check_1_1, 提示="每一句用一個 print()，文字要放在引號裡面。")

def _check_1_2(run):
    out, ns = run()
    lines = 行列表(out)
    if len(lines) < 2: return (False, "要印出兩行。")
    if "2+9-4=" not in _squash(lines[0]): return (False, "第一行要照字面印出 2 + 9 - 4 =")
    return (_squash(lines[1]) == "7", "第二行應該是計算結果 7（算式不要加引號）。")
任務定義("1-2", _check_1_2, 提示="加引號的會照字面印出；不加引號的會被計算。")

def _check_1_3(run):
    out, ns = run()
    return 出現(out, "歡迎來到Python冒險之旅")
任務定義("1-3", _check_1_3, 提示="函式名稱是 print；字串兩邊都要有引號；括號要成對。")

def _check_1_4(run):
    out, ns = run()
    src_lines = [ln.strip() for ln in run.src.splitlines()[1:]]
    has_comment = any(ln.startswith("#") or ("#" in ln and not ln.startswith("print")) for ln in src_lines)
    if not has_comment:
        return (False, "沒有找到以 # 開頭的註解。")
    return 出現(out, "本日行程", "通關密語")
任務定義("1-4", _check_1_4, 提示="在程式行的上面新增一行，以 # 開頭，後面寫說明文字。")


## 🐍 1-1　程式是什麼？Python 又是什麼？
- **程式（program）** = 一連串告訴電腦「做什麼、怎麼做」的指令。
- **Python** 是 1991 年由 Guido van Rossum 發表的語言，語法簡潔、接近英文，是目前**資料分析與 AI 最主流**的語言。
- Python 是 **直譯式（interpreter）** 語言：寫一行就能執行一行，非常適合初學與實驗（Colab 的每一格就是這樣）。
- 課本用的是 Anaconda + Spyder；我們這 6 天改用 **Colab**，觀念完全相同，只是不用安裝。

### 課本 1.3 Python 的特色（濃縮版）
1. 簡單易學、可讀性高（用**縮排**表示程式區塊）
2. 免費、開放原始碼、跨平台
3. 函式庫超多：資料分析（pandas）、繪圖（matplotlib）、機器學習（scikit-learn）……

## 1-2　`print()`：把東西顯示出來
`print()` 是你最常用的指令。括號裡放**字串**（用引號包起來的文字）或**數字／運算式**。

In [ ]:
print('Hello Python!')      # 課本 ex01/first.py
print("單引號或雙引號都可以")
print(2 + 9 - 4)            # 沒有引號 → 當成算式計算
print("2 + 9 - 4")          # 有引號 → 照字面印出
print("台北", "台中", "台南")  # 用逗號分隔多個東西，會自動用空格隔開

## 1-3　註解：寫給人看的說明
`#` 後面的文字 Python 會**完全忽略**，用來解釋程式在做什麼。好的註解讓一週後的你（和隊友）看得懂。

In [ ]:
# 這是一行註解，不會被執行
print("註解不會被印出來")   # 行尾也可以加註解
"""
連續三個引號可以
寫很多行的說明文字
"""

### 🎯 任務 1-1　唐詩產生器

用 **4 個 `print()`** 印出課本第 1 章習題的唐詩，每句一行：

床前明月光，\
疑是地上霜。\
舉頭望明月，\
低頭思故鄉。

**預期結果（範例）**
```
床前明月光，
疑是地上霜。
舉頭望明月，
低頭思故鄉。
```

In [ ]:
# 🎯 任務 1-1　唐詩產生器（請保留這一行）
print("床前明月光，")
# 接著印出其他三句

In [ ]:
檢查("1-1")   # ◀ 執行這一格，看看任務 1-1 有沒有過關

### 🎯 任務 1-2　算式 vs. 字串

請印出 **兩行**：第一行是文字 `2 + 9 - 4 =`（照字面顯示），第二行是**計算結果**（讓 Python 算）。

In [ ]:
# 🎯 任務 1-2　算式 vs. 字串（請保留這一行）
print("2 + 9 - 4 =")
print(???)   # 把 ??? 換成不加引號的算式

In [ ]:
檢查("1-2")   # ◀ 執行這一格，看看任務 1-2 有沒有過關

### 🎯 任務 1-3　抓蟲大隊

下面的程式有 **3 個錯誤**（錯字、少引號、少括號）。請修正它，讓它順利印出 `歡迎來到 Python 冒險之旅`。

先執行一次看看錯誤訊息長什麼樣，再動手修。

In [ ]:
# 🎯 任務 1-3　抓蟲大隊（請保留這一行）
pritn("歡迎來到 Python 冒險之旅)
print("修好我吧！"

In [ ]:
檢查("1-3")   # ◀ 執行這一格，看看任務 1-3 有沒有過關

### 🎯 任務 1-4　加上註解

在下面的程式裡 **至少加一行 `#` 開頭的註解**，說明這段程式在做什麼，並保留原本的輸出。

In [ ]:
# 🎯 任務 1-4　加上註解（請保留這一行）
print("本日行程：Python 概觀 → 變數 → 運算子")
print("目標：拿到 4 個通關密語！")

In [ ]:
檢查("1-4")   # ◀ 執行這一格，看看任務 1-4 有沒有過關

## 💡 挑戰題（不計分）
用 `print()` 和鍵盤符號畫出一隻小貓或一個你喜歡的圖案，例如：
```
 /\_/\
( o.o )
 > ^ <
```

In [ ]:
print(" /\\_/\\ ")
print("( o.o )")
print(" > ^ < ")

---
## 🔑 通關密語

全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：📦 L02 變數與資料型別** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/python-quest-2026/blob/main/notebooks/L02_variables_types.ipynb)

回到入口網頁：https://johnnychao.github.io/python-quest-2026/